# Phase 3 — Render-only

This notebook **renders an existing `output/review/` dossier**. It does
not run the planner or prebuild. Use it for fast iteration on grade,
backplate, cover-pick, and other render-time knobs after you have
already produced a complete review folder.

Inputs required (all under `output/` inside the repo clone after this notebook runs):

| Path                                  | Contents                                          |
| ------------------------------------- | ------------------------------------------------- |
| `output/shot_plan.json`       | Shot plan (from Phase 3 planner)                  |
| `output/narration.mp3`          | Narration audio (from Phase 2)                    |
| `output/bg_music.mp3`          | Music bed (from user)                             |
| `output/review/decisions.json`        | Dossier with chosen candidates per shot           |
| `output/review/shot_*/`               | Per-shot candidate folders                        |
| `output/review/overrides/`            | Optional user overrides (cover, portrait)         |

Set `SOURCE` in cell 1 to either `"zip"` (upload a packed archive) or
`"drive"` (mount Google Drive and copy from a path you specify).


In [1]:
# Used to securely store your API key
from google.colab import userdata
import os

# Set your Hugging Face token from Colab's secrets manager
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN environment variable set.")
print("You may need to restart your runtime for this to take full effect in all processes.")

HF_TOKEN environment variable set.
You may need to restart your runtime for this to take full effect in all processes.


In [2]:
# ════════════════════════════════════════════════════════════════════
# Settings — edit before running.
# ════════════════════════════════════════════════════════════════════
SOURCE = "drive"           # "zip" (upload plan + review/) or "drive"
DRIVE_SOURCE_DIR = "/content/drive/MyDrive/_Phase3/sources"
ARCHIVE_ZIP_NAME = "Archive.zip"

# ── Book Details ────────────────────────────────────────────────────
BOOK_TITLE     = "مذكرات جعفر العسكري"
CHARACTER_NAME = "Jafar al-Askari"

# ── Look ────────────────────────────────────────────────────────────
BOOK_COVER_PICK   = 2          # 1..N over resources/book_cover/
BOOK_COVER_FIT    = "contain"  # fill | contain | blur_pad
BOOK_COVER_ALIGN  = "left"    # center | left | right
TYPOGRAPHY_FAMILY = "B"        # A | B | C
GRADE             = "warm"     # warm | cool | neutral | bw
CAPTION_BACKPLATE = "off"      # off | subtle | solid
TEXT_SCRIM        = "auto"     # auto | off | soft | band  (auto = plate only on bright/busy frames)
OVERLAY_ANCHOR    = "auto"     # auto | center | lower  (auto = quotes/names lower-third, section marks centered)
TITLE_SUBTITLE    = "١٩٣٦ - ١٨٨٥"         # optional sub-line under the main title (author / dates); "" = title only
WORD_REVEAL       = True      # word-by-word reveal on over-image quotes (experimental)
GRADE_MAP         = ""         # optional per-section grading JSON path ({"opening":"neutral",...}); "" = single GRADE
MUSIC_DB          = -12.0      # music bed level in dB (default -18)

# ── Text styling (blank = pipeline default) ─────────────────────────
TITLE_SIZE    = 1.0   # main-title size multiplier (1.2 = 20% larger)
TITLE_COLOR   = ""    # "#RRGGBB" or "" -> family default (aged gold)
CAPTION_SIZE  = 1.5   # caption size multiplier
CAPTION_COLOR = ""    # "#RRGGBB" or "" -> white
CAPTION_POS   = ""    # fraction of height from bottom, e.g. "0.08"; "" -> default
# NOTE: captions are disabled below via --no-captions; the CAPTION_* knobs
# take effect only if you remove that flag in the render cell.

# ── Output Files & Directories ──────────────────────────────────────
OUTPUT_BASE_DIR         = "output"
OUTPUT_FILE             = f"{OUTPUT_BASE_DIR}/final_cut_{TYPOGRAPHY_FAMILY}.mp4"
LOG_FILE                = f"{OUTPUT_BASE_DIR}/render.log"
CONDITIONED_ZIP_FILE    = f"{OUTPUT_BASE_DIR}/conditioned.zip"
RO_ZIP_FILE             = "output_files_ro.zip"
DRIVE_SAVE_RO_DIR       = "/content/drive/MyDrive/_Phase3/output/ro"

# Generated artifacts — supplied at render time (upload .zip or Drive):
PLAN_FILE   = f"{OUTPUT_BASE_DIR}/shot_plan.json"
REVIEW_DIR  = f"{OUTPUT_BASE_DIR}/review/"

# Committed inputs — ship with the repo clone under resources/:
SCRIPT_FILE = "resources/script/main_script.txt"
AUDIO_FILE  = "resources/audio/narration.mp3"
MUSIC_BED   = "resources/audio/bg_music.mp3"

# print(f"SOURCE={SOURCE!r}  OUTPUT={OUTPUT_FILE!r}  GRADE={GRADE!r}  SCRIM={TEXT_SCRIM!r}")
print(f"OUTPUT= {OUTPUT_FILE}\nGRADE = {GRADE}\nSCRIM = {TEXT_SCRIM}\n")

OUTPUT= output/final_cut_B.mp4
GRADE = warm
SCRIM = auto



In [3]:
# ════════════════════════════════════════════════════════════════════
# Bootstrap — clone the Lamahat repo (single source of truth).
# Code, fonts/ and resources/ arrive together at ONE commit, so this
# notebook and the Streamlit app can never drift apart.  Pin BRANCH to
# a feature branch to test unreleased work; leave "main" for releases.
# ════════════════════════════════════════════════════════════════════
REPO   = "https://github.com/abdoljh/Lamahat.git"
BRANCH = "main"

import os, shutil
if os.path.isdir("/content/Lamahat"):
    shutil.rmtree("/content/Lamahat")
!git clone --depth 1 --branch {BRANCH} {REPO} /content/Lamahat
%cd /content/Lamahat
!git log -1 --pretty="✅ Running at commit: %h  %s"


Cloning into '/content/Lamahat'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 211 (delta 17), reused 112 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 64.98 MiB | 43.35 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/Lamahat
✅ Running at commit: 1b571f3  Merge pull request #57 from abdoljh/claude/phase3-movie-quality-d1n58l


In [4]:
# Render-time dependencies. The planner/prebuild are NOT used here, so
# anthropic, pexels, and Whisper are not required. We only need the
# rendering stack: Pillow + Arabic shaping + bidi. Phase3's render.py
# also relies on ffmpeg, which Colab ships preinstalled.
!pip install --quiet pillow arabic-reshaper python-bidi
print("✓ render-time dependencies installed")

# Confirm ffmpeg is on PATH (Colab default — sanity check only)
!ffmpeg -version 2>&1 | head -1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 11.3 MB/s eta 0:00:00
✓ render-time dependencies installed
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


In [5]:
import os, shutil, subprocess
from pathlib import Path

CONTENT = Path.cwd()   # the repo clone (/content/Lamahat on Colab)
OUT = CONTENT / OUTPUT_BASE_DIR
OUT.mkdir(parents=True, exist_ok=True)

def _confirm_inputs():
    plan      = OUT / Path(PLAN_FILE).name
    audio     = CONTENT / AUDIO_FILE
    music     = CONTENT / MUSIC_BED
    review    = Path(REVIEW_DIR)
    decisions = review / "decisions.json"
    print("\nChecking inputs:")
    for label, p in [("plan", plan), ("audio", audio), ("music", music),
                     ("review/", review), ("decisions", decisions)]:
        print(f"  {chr(0x2713) if p.exists() else chr(0x2717)} {label:10s} {p}")
    missing = [p for p in (plan, audio, review, decisions) if not p.exists()]
    assert not missing, f"Missing required inputs: {missing}"
    print("\n\u2705 All inputs present.")

if SOURCE == "zip":
    # Upload one .zip that extracts under /content/output/ to give
    # plan + review/.  (audio/music now ship with the repo clone.)
    try:
        from google.colab import files   # type: ignore
        print("Upload one .zip containing plan + review/ ...")
        uploaded = files.upload(); assert uploaded, "No file uploaded"
        zip_name = next(iter(uploaded))
    except ImportError:
        zip_name = ARCHIVE_ZIP_NAME # Use ARCHIVE_ZIP_NAME
        assert Path(zip_name).exists(), f"Outside Colab - put {ARCHIVE_ZIP_NAME} at /content/{ARCHIVE_ZIP_NAME}"
    print(f"Extracting {zip_name} into {OUT} ...")
    subprocess.run(["unzip", "-oq", zip_name, "-d", str(OUT)], check=True)
    # auto-flatten a single wrapping dir (e.g. zip wrapped in output/)
    top = list(OUT.iterdir())
    if len(top) == 1 and top[0].is_dir() and (top[0] / "review").is_dir():
        inner = top[0]; print(f"Flattening {inner.name}/")
        for child in inner.iterdir():
            shutil.move(str(child), str(OUT / child.name))
        inner.rmdir()

elif SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    src = Path(DRIVE_SOURCE_DIR); assert src.is_dir(), f"Not found: {src}"

    # Path to the zip file in Google Drive
    zip_path = src / ARCHIVE_ZIP_NAME
    assert zip_path.exists(), f"Expected zip file at {zip_path}"
    print(f"Found zip file: {zip_path}")

    # Extract the zip file into OUT directory
    print(f"Extracting {zip_path.name} into {OUT} ...")
    subprocess.run(["unzip", "-oq", str(zip_path), "-d", str(OUT)], check=True)
    print("Extraction complete.")

    # auto-flatten a single wrapping dir (e.g. zip wrapped in output/)
    top = list(OUT.iterdir())
    if len(top) == 1 and top[0].is_dir() and (top[0] / "review").is_dir():
        inner = top[0]; print(f"Flattening {inner.name}/")
        for child in inner.iterdir():
            shutil.move(str(child), str(OUT / child.name))
        inner.rmdir()

else:
    raise ValueError(f"SOURCE must be 'zip' or 'drive', got {SOURCE!r}")

# --- FIX START ---
expected_plan_filename = Path(PLAN_FILE).name # This is 'shot_plan.json'
expected_plan_path_in_out = OUT / expected_plan_filename

# Always print debug info to understand the state
print(f"DEBUG (Pre-Fix): Target plan file path: '{expected_plan_path_in_out}'")
print(f"DEBUG (Pre-Fix): Does target plan file exist? {expected_plan_path_in_out.exists()}")

# The core problem is that _confirm_inputs reports it missing.
# We will always attempt to find and move it if it's not at the root level of OUT.
if not expected_plan_path_in_out.exists():
    print(f"'{expected_plan_filename}' is NOT at the expected root '{OUT}'. Searching recursively...")
    found_plan_files = list(OUT.rglob(expected_plan_filename))
    print(f"DEBUG: Recursive search for '{expected_plan_filename}' found: {found_plan_files}")

    if found_plan_files:
        found_plan_path = found_plan_files[0]
        if found_plan_path != expected_plan_path_in_out: # Only move if it's not already there
            print(f"Found '{expected_plan_filename}' at '{found_plan_path}'. Moving to '{expected_plan_path_in_out}'...")
            try:
                shutil.move(str(found_plan_path), str(expected_plan_path_in_out))
                print(f"Successfully moved '{expected_plan_filename}'.")
            except Exception as e:
                print(f"ERROR moving '{found_plan_path}' to '{expected_plan_path_in_out}': {e}")
        else:
            print(f"'{expected_plan_filename}' was found at '{found_plan_path}' but it's already the target path. No move needed.")
    else:
        print(f"WARNING: '{expected_plan_filename}' was NOT found anywhere under '{OUT}'.")
else:
    print(f"'{expected_plan_filename}' already exists at '{expected_plan_path_in_out}'. Skipping search and move.")

print(f"DEBUG (Post-Fix): Does target plan file exist? {expected_plan_path_in_out.exists()}")
# --- FIX END ---

_confirm_inputs()


Mounted at /content/drive
Found zip file: /content/drive/MyDrive/_Phase3/sources/Archive.zip
Extracting Archive.zip into /content/Lamahat/output ...
Extraction complete.
DEBUG (Pre-Fix): Target plan file path: '/content/Lamahat/output/shot_plan.json'
DEBUG (Pre-Fix): Does target plan file exist? False
'shot_plan.json' is NOT at the expected root '/content/Lamahat/output'. Searching recursively...
DEBUG: Recursive search for 'shot_plan.json' found: [PosixPath('/content/Lamahat/output/review/shot_plan.json')]
Found 'shot_plan.json' at '/content/Lamahat/output/review/shot_plan.json'. Moving to '/content/Lamahat/output/shot_plan.json'...
Successfully moved 'shot_plan.json'.
DEBUG (Post-Fix): Does target plan file exist? True

Checking inputs:
  ✓ plan       /content/Lamahat/output/shot_plan.json
  ✓ audio      /content/Lamahat/resources/audio/narration.mp3
  ✓ music      /content/Lamahat/resources/audio/bg_music.mp3
  ✓ review/    output/review
  ✓ decisions  output/review/decisions.json



In [6]:
!python condition_assets.py --review-dir {REVIEW_DIR} # --sr realesrgan]; --dry-run

INFO  condition_assets  Shot 2 contain    sr 935x1200 -> 1247x1600 [sr]
INFO  condition_assets  Shot 4 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 5 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 6 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 8 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 10 toned (documentary palette, source=pexels)
INFO  condition_assets  Shot 10 cover_crop upscale 1880x1253 -> 2560x1440 [upscaled]  crop=(0, 195, 1880, 1253)
INFO  condition_assets  Shot 11 toned (documentary palette, source=pexels)
INFO  condition_assets  Shot 11 cover_crop upscale 1880x1253 -> 2560x1440 [upscaled]  crop=(0, 0, 1880, 1058)
INFO  condition_assets  Shot 13 cover_crop upscale 1280x894 -> 2560x1440 [upscaled]  crop=(0, 142, 1280, 862)
INFO  condition_assets  Shot 14 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 15 cover      asis 2560x1440 -> 2560x1440

In [7]:
# Zip conditioned assets (OPTIONAL)
import os
import zipfile

output_dir = REVIEW_DIR
zip_filename = CONDITIONED_ZIP_FILE

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    files_added = []
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure relative to 'output_dir'
            # Skip .mp3 files if any exist (though unlikely in review folder)
            if not file_path.endswith('.mp3'):
                zipf.write(file_path, os.path.relpath(file_path, output_dir))
                files_added.append(file_path)

print(f"🤐 Successfully created '{zip_filename}' containing: ")
if files_added:
    for f in files_added:
        print(f"  - {f}")
else:
    print("⚠️ No files found to zip in the", REVIEW_DIR, "directory!")


🤐 Successfully created 'output/conditioned.zip' containing: 
  - output/review/decisions.json
  - output/review/script.txt
  - output/review/planner_raw_response.txt
  - output/review/word_timings.json
  - output/review/photo_bank_assignment_raw.txt
  - output/review/README.txt
  - output/review/shot_42_typography/card_preview.png
  - output/review/shot_42_typography/context.txt
  - output/review/shot_22_archive/candidates.json
  - output/review/shot_22_archive/bank_army_officers_at_palace*.jpg
  - output/review/shot_22_archive/context.txt
  - output/review/shot_20_typography/card_preview.png
  - output/review/shot_20_typography/context.txt
  - output/review/shot_60_archive/candidates.json
  - output/review/shot_60_archive/bank_arab_revolt_2.jpeg
  - output/review/shot_60_archive/context.txt
  - output/review/shot_80_typography/card_preview.png
  - output/review/shot_80_typography/context.txt
  - output/review/shot_37_portrait/candidates.json
  - output/review/shot_37_portrait/context.

In [9]:
!python regenerate_captions.py --review-dir {REVIEW_DIR}

INFO existing plan: 80 caption events, 0 already carry word timing
INFO Section detection: 4 boundaries (point_1, point_2, point_3, closing)
INFO regenerated 80 caption events, all carrying word timing
INFO wrote output/review/shot_plan.json (84 shots, 80 captions)


In [10]:
# ════════════════════════════════════════════════════════════════════
# Render
# ════════════════════════════════════════════════════════════════════
import shlex, subprocess
from pathlib import Path

Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

cmd = [
    "python", "render_plan.py",
    "--plan",              PLAN_FILE,
    "--audio",             AUDIO_FILE,
    "--music",             MUSIC_BED,
    "--music-gain",        str(MUSIC_DB),
    "--review-dir",        REVIEW_DIR,
    "--book-cover-pick",   str(BOOK_COVER_PICK),
    "--book-cover-fit",    BOOK_COVER_FIT,
    "--book-cover-align",  BOOK_COVER_ALIGN,
    "--typography-family", TYPOGRAPHY_FAMILY,
    "--parallax",
    "--typography-over-image",

    "--grade",             GRADE,
    "--caption-backplate", CAPTION_BACKPLATE,
    "--text-scrim",        TEXT_SCRIM,
    "--overlay-anchor",    OVERLAY_ANCHOR,
    "--title-subtitle",    TITLE_SUBTITLE,
    *(["--word-reveal"] if WORD_REVEAL else []),
    *(["--grade-map", GRADE_MAP] if GRADE_MAP else []),
    "--title-size",        str(TITLE_SIZE),
    "--caption-size",      str(CAPTION_SIZE),
    "--output",            OUTPUT_FILE,
]
# Optional colour / position flags — only sent when explicitly set above.
if TITLE_COLOR:   cmd += ["--title-color",   TITLE_COLOR]
if CAPTION_COLOR: cmd += ["--caption-color", CAPTION_COLOR]
if CAPTION_POS:   cmd += ["--caption-pos",   str(CAPTION_POS)]

print("Running:\n  " + " ".join(shlex.quote(c) for c in cmd) + "\n")

# Backgrounded so the next cell can tail render.log.
with open(LOG_FILE, "w") as log: # Use LOG_FILE
    subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print(f"Rendering begins ... tail {LOG_FILE} with the next cell.")

Running:
  python render_plan.py --plan output/shot_plan.json --audio resources/audio/narration.mp3 --music resources/audio/bg_music.mp3 --music-gain -12.0 --review-dir output/review/ --book-cover-pick 2 --book-cover-fit contain --book-cover-align left --typography-family B --parallax --typography-over-image --grade warm --caption-backplate off --text-scrim auto --overlay-anchor auto --title-subtitle '١٩٣٦ - ١٨٨٥' --word-reveal --title-size 1.0 --caption-size 1.5 --output output/final_cut_B.mp4

Rendering begins ... tail output/render.log with the next cell.


In [11]:
# Monitor rendering progress
import time
from IPython.display import clear_output

log_path = LOG_FILE

print("Monitoring rendering progress...")
while True:
    try:
        # Read the log file contents
        try:
            with open(log_path, "r") as f:
                log_content = f.read()
        except FileNotFoundError:
            log_content = ""

        # Clear cell output and show the last 20 lines
        clear_output(wait=True)
        lines = log_content.splitlines()
        print("\n".join(lines[-20:]))

        # Check if the script's success signature is in the log
        if "Done in" in log_content or "Rendered video →" in log_content:
            print("\n✅ Rendering process completed successfully! Stopped monitoring.")
            break

        time.sleep(5)

    except KeyboardInterrupt:
        print("\n⚠️ Monitoring stopped manually. The script may still be running!")
        break


INFO  phase3.render  [render 71%] shot 79/84: broll
INFO  phase3.sources.decisions  Shot 79: chosen-file hit bank_jafar_sovereign_nation*.jpg
INFO  phase3.render  [render 72%] shot 80/84: typography
INFO  phase3.sources.decisions  Shot 79: chosen-file hit bank_jafar_sovereign_nation*.jpg
INFO  phase3.render  [render 72%] shot 81/84: typography
INFO  phase3.sources.decisions  Shot 79: chosen-file hit bank_jafar_sovereign_nation*.jpg
INFO  phase3.render  [render 73%] shot 82/84: typography
INFO  phase3.sources.decisions  Shot 79: chosen-file hit bank_jafar_sovereign_nation*.jpg
INFO  phase3.render  [render 74%] shot 83/84: typography
INFO  phase3.sources.decisions  Shot 79: chosen-file hit bank_jafar_sovereign_nation*.jpg
INFO  phase3.render  [render 75%] shot 84/84: typography
INFO  phase3.render  [render 80%] concat all shots
INFO  phase3.render  [render 86%] generating captions
INFO  phase3.render  [render 92%] mux audio and captions
INFO  phase3.render  Color grade applied: warm
INFO

In [12]:
# Zip output files for exporting
import os
import zipfile

output_dir = OUTPUT_BASE_DIR
zip_filename = RO_ZIP_FILE

# Get all files in the output directory
files_to_zip = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f))]

# Filter out the .mp3 file
filtered_files = [f for f in files_to_zip if not f.endswith('.mp3')]

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in filtered_files:
        # Add file to zip, preserving directory structure relative to 'output_dir'
        zipf.write(file_path, os.path.relpath(file_path, output_dir))

print(f"🤐 Successfully created '{zip_filename}' containing: ")
for f in filtered_files:
    print(f"  - {f}")

🤐 Successfully created 'output_files_ro.zip' containing: 
  - output/final_cut_B.mp4
  - output/render.log
  - output/main_script.txt
  - output/conditioned.zip
  - output/shot_plan.json


In [13]:
# Save zipped file to Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Define the destination directory and file path
dest_dir = DRIVE_SAVE_RO_DIR
dest_file_path = os.path.join(dest_dir, RO_ZIP_FILE) # Use RO_ZIP_FILE

# Create the destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)
print(f"Destination directory '{dest_dir}' ensured to exist.")

# --- Test write access to the directory ---
test_file = os.path.join(dest_dir, 'test_write.txt')
try:
    with open(test_file, 'w') as f:
        f.write('This is a test file.\n')
    print(f"✅ Successfully wrote test file to '{test_file}'.")
    os.remove(test_file) # Clean up the test file
    print(f"Test file '{test_file}' removed.")
except Exception as e:
    print(f"Error writing test file to '{test_file}': {e}")
    print("⚠️ It seems there might be a permissions or access issue with Google Drive.")
    # Exit or raise an error if write access fails
    raise
# ----------------------------------------

shutil.copy(RO_ZIP_FILE, dest_file_path)
print(f"📽️ Files are saved to Google Drive at '{dest_file_path}'.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Destination directory '/content/drive/MyDrive/_Phase3/output/ro' ensured to exist.
✅ Successfully wrote test file to '/content/drive/MyDrive/_Phase3/output/ro/test_write.txt'.
Test file '/content/drive/MyDrive/_Phase3/output/ro/test_write.txt' removed.
📽️ Files are saved to Google Drive at '/content/drive/MyDrive/_Phase3/output/ro/output_files_ro.zip'.
